<a href="https://colab.research.google.com/github/alimkacar/google-play-scraper-and-nlp/blob/main/google_play_scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
# Gerekli paket
!pip install google-play-scraper pandas
!pip install pandas nltk regex
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
import re
import nltk
from tensorflow.keras.preprocessing.text import Tokenizer
from google_play_scraper import reviews
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split


def fetch_reviews(app_id, lang='tr', country='tr', pages=30):
    """
    Uygulama yorumlarını sayfa sayısı kadar çekip DataFrame döndürür.
    """
    all_reviews = []
    token = None

    for i in range(pages):
        r, token = reviews(
            app_id,
            lang=lang,
            country=country,
            count=300,
            continuation_token=token
        )
        all_reviews.extend(r)
        if not token:
            break

    df = pd.DataFrame(all_reviews)
    df = df.rename(columns={'content':'text','score':'rating'})
    keep_cols = ['text','rating','at','userName','thumbsUpCount']
    df = df[[c for c in keep_cols if c in df.columns]]
    df['app_id'] = app_id
    return df


apps = {
    'WhatsApp': 'com.whatsapp',
    'TikTok': 'com.zhiliaoapp.musically',
    'X (Twitter)': 'com.twitter.android',
    'Instagram': 'com.instagram.android',
    'Facebook': 'com.facebook.katana',
    'Telegram': 'org.telegram.messenger',
}

all_dfs = []

for name, app_id in apps.items():
    print(f"Çekiliyor: {name}")
    df = fetch_reviews(app_id, lang='tr', country='tr', pages=30)
    all_dfs.append(df)


final = pd.concat(all_dfs, ignore_index=True)

final=final.drop(["at", "userName", "thumbsUpCount", "app_id"], axis=1)


final.to_csv('google_play_reviews_apps.csv', index=False, encoding='utf-8-sig')
print("Toplam yorum sayısı:", final.shape[0])
final.head()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Çekiliyor: WhatsApp
Çekiliyor: TikTok
Çekiliyor: X (Twitter)
Çekiliyor: Instagram
Çekiliyor: Facebook
Çekiliyor: Telegram
Toplam yorum sayısı: 54000


,text,rating
0,uygulama süper yapay zekası var çok rahat kull...,5
1,faydalı olmadı msj gonderemedim şifre alabilme...,1
2,mükemmel tabii ki.Çok güzel,5
3,çok güzel bir uygulama,5
4,vasap,5


In [42]:

stop_words = set([
    "ve","ile","bu","ben","sen","o","biz","siz","onlar","için","çok","gibi",
    "the","is","in","it","this","that","a","an","of","on","at","to","and","or"
])


def lower_text(text):
    return text.lower()


def remove_noise(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z0-9ğüşöçıİĞÜŞÖÇ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def remove_stopwords(text):
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)


def clean_text(text):
    text = lower_text(text)
    text = remove_noise(text)
    text = remove_stopwords(text)
    return text


In [43]:
final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54000 entries, 0 to 53999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    53998 non-null  object
 1   rating  54000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 843.9+ KB


In [44]:

df = pd.read_csv('google_play_reviews_apps.csv')


df['text_clean'] = df['text'].astype(str).apply(clean_text)


df[['text', 'text_clean', 'rating']].head(10)


df.to_csv('google_play_reviews_clean.csv', index=False, encoding='utf-8-sig')


In [45]:
df2=pd.read_csv("google_play_reviews_clean.csv")
df2.info()

df2=df2.drop(["text"],axis=1)
df2.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54000 entries, 0 to 53999
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text        53998 non-null  object
 1   rating      54000 non-null  int64 
 2   text_clean  52455 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.2+ MB


,rating,text_clean
0,5,uygulama süper yapay zekası var rahat kullanıy...
1,1,faydalı olmadı msj gonderemedim şifre alabilmek
2,5,mükemmel tabii ki güzel
3,5,güzel bir uygulama
4,5,vasap


In [46]:

tokenizer=Tokenizer(
    num_words=None,
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
    lower=True,
    split=' ',
    char_level=False,
    oov_token=None,
    analyzer=None,
)

In [47]:
target = df2["rating"].values.tolist()
data = df2["text_clean"].values.tolist()

In [50]:

data = [item if isinstance(item, str) else "" for item in data]

tokenizer.fit_on_texts(data)
sequences = tokenizer.texts_to_sequences(data)
X = pad_sequences(sequences, maxlen=100, padding='post')

In [52]:
import pickle

print(f"Orijinal metin: {data[0]}")
print(f"Sequence: {sequences[0][:10]}")  # İlk 10 sayı
print(f"Padded: {X[0][:10]}")
print(f"\nX shape: {X.shape}")


with open("tokenizer.pickle", "wb") as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("✓ Tokenizer kaydedildi!")

Orijinal metin: uygulama süper yapay zekası var rahat kullanıyorum fakat bende bir sorun var dayım bana yazıyor mu yazmıyor mu onu göstermiyo babamda da aynı yapay zeka argolu kelimeler kullanıyo bağırıyo ama bunlardan puan kırmayacağım teşekkürler
Sequence: [2, 77, 480, 4897, 8, 811, 175, 155, 176, 1]
Padded: [   2   77  480 4897    8  811  175  155  176    1]

X shape: (54000, 100)
✓ Tokenizer kaydedildi!


In [56]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

print("\n" + "="*60)
print("3. TARGET HAZIRLAMA")
print("="*60)


y = np.array([1 if rating >= 4 else 0 for rating in target])

pozitif_sayisi = y.sum()
negatif_sayisi = len(y) - pozitif_sayisi

print(f"✓ Pozitif (4-5 yıldız): {pozitif_sayisi} (%{pozitif_sayisi/len(y)*100:.1f})")
print(f"✓ Negatif (1-3 yıldız): {negatif_sayisi} (%{negatif_sayisi/len(y)*100:.1f})")


print("\n" + "="*60)
print("4. VERİYİ AYIRMA")
print("="*60)


MAX_LEN = 100

MAX_WORDS = len(tokenizer.word_index) + 1
print(f"✓ Kelime Hazinesi (Vocab Size): {MAX_WORDS}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ Train: {X_train.shape[0]} örnек")
print(f"✓ Test:  {X_test.shape[0]} örnek")


print("\n" + "="*60)
print("5. MODEL OLUŞTURMA")
print("="*60)

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


print("\n" + "="*60)
print("6. MODEL EĞİTİMİ")
print("="*60)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=64,
    verbose=1
)


print("\n" + "="*60)
print("7. MODEL DEĞERLENDİRME")
print("="*60)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"✓ Test Accuracy: {test_acc*100:.2f}%")
print(f"✓ Test Loss: {test_loss:.4f}")


print("\n" + "="*60)
print("8. MODEL KAYDETME")
print("="*60)

model.save('sentiment_model.h5')
print("✓ Model kaydedildi: sentiment_model.h5")


print("\n" + "="*60)
print("9. TAHMİN FONKSİYONU TEST")
print("="*60)


stop_words = set([
    "ve","ile","bu","ben","sen","o","biz","siz","onlar","için","çok","gibi",
    "the","is","in","it","this","that","a","an","of","on","at","to","and","or"
])

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z0-9ğüşöçıİĞÜŞÖÇ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

def predict_sentiment(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    score = model.predict(padded, verbose=0)[0][0]

    return {
        'metin': text,
        'pozitif_yuzde': round(score * 100, 2),
        'negatif_yuzde': round((1-score) * 100, 2),
        'sonuc': '😊 Pozitif' if score >= 0.5 else '😞 Negatif',
        'guven': round(abs(score - 0.5) * 200, 2)
    }


test_metinler = [
    "Bu uygulama gerçekten harika, çok beğendim!",
    "Berbat bir uygulama, hiç beğenmedim.",
    "İdare eder, ortalama bir deneyim.",
    "Mükemmel! Kesinlikle tavsiye ederim.",
    "Çok kötü, para kaybı."
]

print("\nÖrnek Tahminler:")
print("-"*60)

for metin in test_metinler:
    result = predict_sentiment(metin)
    print(f"\n📝 Metin: {result['metin']}")
    print(f"   Pozitif: %{result['pozitif_yuzde']} | Negatif: %{result['negatif_yuzde']}")
    print(f"   Sonuç: {result['sonuc']} (Güven: %{result['guven']})")

print("\n" + "="*60)
print("✅ TÜM İŞLEMLER TAMAMLANDI!")
print("="*60)
print("\nKaydedilen dosyalar:")
print("  • sentiment_model.h5")
print("  • tokenizer.pickle")
print("\nBu dosyaları kullanarak yeni metinler üzerinde tahmin yapabilirsiniz!")


3. TARGET HAZIRLAMA
✓ Pozitif (4-5 yıldız): 30373 (%56.2)
✓ Negatif (1-3 yıldız): 23627 (%43.8)

4. VERİYİ AYIRMA
✓ Kelime Hazinesi (Vocab Size): 58918
✓ Train: 43200 örnек
✓ Test:  10800 örnek

5. MODEL OLUŞTURMA


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


6. MODEL EĞİTİMİ
Epoch 1/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 138s 198ms/step - accuracy: 0.5516 - loss: 0.6878 - val_accuracy: 0.5625 - val_loss: 0.6853
Epoch 2/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 146s 204ms/step - accuracy: 0.5645 - loss: 0.6856 - val_accuracy: 0.5625 - val_loss: 0.6856
Epoch 3/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 135s 200ms/step - accuracy: 0.5651 - loss: 0.6848 - val_accuracy: 0.5625 - val_loss: 0.6854
Epoch 4/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 141s 198ms/step - accuracy: 0.5633 - loss: 0.6855 - val_accuracy: 0.5625 - val_loss: 0.6858
Epoch 5/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 141s 197ms/step - accuracy: 0.5602 - loss: 0.6862 - val_accuracy: 0.5625 - val_loss: 0.6853

7. MODEL DEĞERLENDİRME


✓ Test Accuracy: 56.25%
✓ Test Loss: 0.6853

8. MODEL KAYDETME
✓ Model kaydedildi: sentiment_model.h5

9. TAHMİN FONKSİYONU TEST

Örnek Tahminler:
------------------------------------------------------------

📝 Metin: Bu uygulama gerçekten harika, çok beğendim!
   Pozitif: %56.13999938964844 | Negatif: %43.86000061035156
   Sonuç: 😊 Pozitif (Güven: %12.279999732971191)

📝 Metin: Berbat bir uygulama, hiç beğenmedim.
   Pozitif: %56.13999938964844 | Negatif: %43.86000061035156
   Sonuç: 😊 Pozitif (Güven: %12.279999732971191)

📝 Metin: İdare eder, ortalama bir deneyim.
   Pozitif: %56.13999938964844 | Negatif: %43.86000061035156
   Sonuç: 😊 Pozitif (Güven: %12.279999732971191)

📝 Metin: Mükemmel! Kesinlikle tavsiye ederim.
   Pozitif: %56.13999938964844 | Negatif: %43.86000061035156
   Sonuç: 😊 Pozitif (Güven: %12.279999732971191)

📝 Metin: Çok kötü, para kaybı.
   Pozitif: %56.13999938964844 | Negatif: %43.86000061035156
   Sonuç: 😊 Pozitif (Güven: %12.279999732971191)

✅ TÜM İŞLEMLER TA